# 05 — Targeting Policy, CUPED, and Power Analysis

This notebook turns τ̂(x) into a business decision and shows the experimentation platform thinking.

**The headline sentence:** "Targeting the top 30% by predicted uplift captures ~X% of incremental visits at 30% of spend."

This exact framing is what a DS/experimentation PM wants.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from src.data import load_data, split, get_Xyt, FEATURE_COLS
from src.metrics import evaluate_all, qini_curve
from src.policy import targeting_curve, sleeping_dogs_analysis, policy_summary
from src.cuped import cuped_ate, variance_reduction
from src.power import sample_size, mde, cuped_sample_size, power_table

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

In [ ]:
# Load best model predictions
with open('../data/meta_learner_predictions.pkl', 'rb') as f:
    saved = pickle.load(f)

predictions = saved['predictions']
y_te = saved['y_te']
t_te = saved['t_te']

# pick best model by Qini coeff
from src.metrics import qini_coefficient
best_name = max(predictions, key=lambda k: qini_coefficient(y_te, predictions[k], t_te))
tau_best = predictions[best_name]
print(f'Best model: {best_name}  Qini={qini_coefficient(y_te, tau_best, t_te):.4f}')

## 1. Targeting Policy: Incremental captured vs Spend

In [ ]:
fractions, pct_captured, total_incremental = targeting_curve(y_te, tau_best, t_te)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fractions * 100, pct_captured * 100, linewidth=2, color='#1f77b4', label=best_name)
ax.plot([0, 100], [0, 100], 'k--', linewidth=0.8, label='Random (proportional targeting)')
ax.set_xlabel('% of population treated (spend proxy)')
ax.set_ylabel('% of total incremental visits captured')
ax.set_title('Targeting Efficiency: Incremental Captured vs. Spend')

# annotate key thresholds
for thr in [0.10, 0.20, 0.30, 0.50]:
    idx = np.searchsorted(fractions, thr)
    idx = min(idx, len(pct_captured)-1)
    y_val = pct_captured[idx] * 100
    ax.annotate(f'{thr*100:.0f}% spend → {y_val:.0f}% incr.',
                xy=(thr*100, y_val), xytext=(thr*100+5, y_val-8),
                fontsize=8.5, color='darkblue',
                arrowprops=dict(arrowstyle='->', color='darkblue', lw=0.8))

ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/targeting_policy.png', bbox_inches='tight')
plt.show()

In [ ]:
print('Targeting policy summary:')
for row in policy_summary(y_te, tau_best, t_te, [0.10, 0.20, 0.30, 0.50]):
    print(f"  Top {row['spend_fraction']*100:.0f}% by uplift: "
          f"{row['pct_incremental_captured']:.1f}% of incremental visits")

## 2. Sleeping Dogs: the case for exclusion

In [ ]:
sd = sleeping_dogs_analysis(tau_best, t_te, y_te)
print('Sleeping Dogs analysis:')
for k, v in sd.items():
    print(f'  {k}: {v}')
print()
print('These users have *negative* predicted uplift — advertising hurts them.')
print('Excluding them improves both incrementality AND reduces wasted spend.')

## 3. CUPED — Variance Reduction on ATE

In [ ]:
df = load_data(percent10=True)
train, val, test = split(df)
X_te_raw, y_te_raw, t_te_raw = get_Xyt(test)

# CUPED proxy: use control-arm visit predictions from a control-trained model
# (simulates a pre-experiment period covariate)
from lightgbm import LGBMClassifier
X_tr, y_tr, t_tr = get_Xyt(train)

ctrl_model = LGBMClassifier(n_estimators=200, num_leaves=31, n_jobs=-1, verbose=-1, random_state=42)
ctrl_model.fit(X_tr[t_tr == 0], y_tr[t_tr == 0])
x_pre = ctrl_model.predict_proba(X_te_raw)[:, 1]

result = cuped_ate(y_te_raw, t_te_raw, x_pre)

print('=== CUPED Variance Reduction ===')
print(f'\nRaw ATE:')
print(f'  ATE = {result["raw"]["ate"]:.5f}')
print(f'  95% CI = [{result["raw"]["ci_lo"]:.5f}, {result["raw"]["ci_hi"]:.5f}]')
print(f'  CI width = {result["raw"]["ci_hi"] - result["raw"]["ci_lo"]:.5f}')
print(f'\nCUPED-adjusted ATE:')
print(f'  ATE = {result["cuped"]["ate"]:.5f}')
print(f'  95% CI = [{result["cuped"]["ci_lo"]:.5f}, {result["cuped"]["ci_hi"]:.5f}]')
print(f'  CI width = {result["cuped"]["ci_hi"] - result["cuped"]["ci_lo"]:.5f}')
print(f'\nρ (corr between proxy and outcome): {result["rho"]:.4f}')
print(f'ρ²: {result["rho_squared"]:.4f}')
print(f'Variance reduction: {result["variance_reduction_pct"]:.1f}%')
print(f'CI width reduction: {result["ci_width_reduction_pct"]:.1f}%')
print(f'\nHonesty caveat: Criteo has no true pre-experiment period. This proxy (control-arm model)')
print(f'satisfies independence from T only if fit on a held-out subset.')

## 4. Power Analysis

In [ ]:
p_base = float(y_te_raw[t_te_raw == 0].mean())
print(f'Control arm visit rate (base): {p_base:.4%}')
print()

# Power table
rows = power_table(p_base, [0.001, 0.002, 0.003, 0.005, 0.010])
pt = pd.DataFrame(rows)
pt['mde_pct'] = pt['mde'] * 100
print('Sample size required for various MDEs (alpha=0.05, power=0.80):')
print(pt[['mde_pct', 'n_per_arm', 'total_n']].rename(
    columns={'mde_pct': 'MDE (%)', 'n_per_arm': 'N/arm', 'total_n': 'Total N'}
).to_string(index=False))
print()

# CUPED impact on sample size
rho_sq = result['rho_squared']
r = cuped_sample_size(p_base, 0.003, rho_sq)
print(f'CUPED impact for MDE=0.3pp (rho^2={rho_sq:.3f}):')
print(f'  Without CUPED: {r["n_raw"]:,}/arm')
print(f'  With CUPED:    {r["n_cuped"]:,}/arm  ({r["reduction_pct"]:.1f}% fewer users for same power)')

In [ ]:
# Power curve visualization
mde_vals = np.linspace(0.001, 0.015, 60)
n_raw_vals = [sample_size(p_base, d) for d in mde_vals]
n_cuped_vals = [sample_size(p_base, d) * (1 - rho_sq) for d in mde_vals]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mde_vals * 100, np.array(n_raw_vals) / 1e6, label='Without CUPED', linewidth=2)
ax.plot(mde_vals * 100, np.array(n_cuped_vals) / 1e6, label=f'With CUPED (ρ²={rho_sq:.2f})',
        linewidth=2, linestyle='--')
ax.set_xlabel('Minimum Detectable Effect (% visit rate lift)')
ax.set_ylabel('Required per-arm sample size (millions)')
ax.set_title('Power Analysis: Sample Size vs. MDE\n(CUPED reduces required N by ρ² fraction)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/power_analysis.png', bbox_inches='tight')
plt.show()